This restricts it to just canopy height > 0

In [ ]:
gdalbuildvrt \
  -overwrite \
  -srcnodata 255 \
  -vrtnodata 0 \
  ETH_GlobalCanopyHeight_10m_2020_positive_only.vrt \
  3deg_cogs/*_Map.tif

This then aggregates into 0.1 degree pixels, helps us save space

In [ ]:
time gdalwarp \
  -of netCDF \
  -t_srs EPSG:4326 \
  -te -180 -90 180 90 \
  -tr 0.1 0.1 \
  -tap \
  -r average \
  -ovr NONE \
  -dstnodata -9999 \
  -ot Float32 \
  -overwrite \
  ETH_GlobalCanopyHeight_10m_2020_positive_only.vrt \
  ETH_GlobalCanopyHeight_2020_0p1deg.nc

## LAI

In [3]:
from pathlib import Path
from collections import defaultdict, Counter
from getpass import getpass
import gc
import os
import re

import boto3
from botocore.config import Config
import requests
import rasterio
from rasterio.transform import from_origin
from rasterio.warp import reproject, Resampling
import numpy as np
from netCDF4 import Dataset


# ============================================================
# SETTINGS
# ============================================================

DATASET_ID = "lai_global_1km_10daily_v2"

START_DATE = "2018-01-01T00:00:00.000Z"
END_DATE = "2021-01-01T00:00:00.000Z"  # Exclusive

OUTPUT_RESOLUTION = 0.1

DOWNLOAD_DIR = Path(
    "../../scratch/LAI/lai_global_1km_10daily_v2/2018_2020"
)

OUTPUT_PATH = Path(
    "../../scratch/LAI/"
    "VAI_monthly_climatology_2018_2020_global_0p1deg.nc"
)

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)


# ============================================================
# QUERY THE COPERNICUS CATALOGUE
# ============================================================

catalogue_url = (
    "https://catalogue.dataspace.copernicus.eu/"
    "odata/v1/Products"
)

odata_filter = (
    "Collection/Name eq 'CLMS' and "
    "Attributes/OData.CSC.StringAttribute/any("
    "att:att/Name eq 'datasetIdentifier' and "
    f"att/OData.CSC.StringAttribute/Value eq '{DATASET_ID}') and "
    "Attributes/OData.CSC.StringAttribute/any("
    "att:att/Name eq 'fileFormat' and "
    "att/OData.CSC.StringAttribute/Value eq 'cog') and "
    f"ContentDate/Start ge {START_DATE} and "
    f"ContentDate/Start lt {END_DATE}"
)

params = {
    "$filter": odata_filter,
    "$select": "Id,Name,S3Path,ContentDate",
    "$orderby": "ContentDate/Start asc",
    "$top": 1000,
}

products = []
url = catalogue_url

while url:
    response = requests.get(
        url,
        params=params if url == catalogue_url else None,
        timeout=120,
    )
    response.raise_for_status()

    result = response.json()
    products.extend(result["value"])
    url = result.get("@odata.nextLink")

print(f"Catalogue returned {len(products)} products")


# ============================================================
# SELECT THE BEST LAI PRODUCT FOR EACH DATE
# ============================================================

def is_lai_product(product):
    """
    Accept:
      c_gls_LAI_YYYYMMDDHHMM_...
      c_gls_LAI-RT0_YYYYMMDDHHMM_...
      c_gls_LAI-RT1_YYYYMMDDHHMM_...
      etc.

    Exclude RMSE, QFLAG, NOBS and other ancillary products.
    """
    return bool(
        re.search(
            r"^c_gls_LAI(?:-RT\d+)?_\d{12}_GLOBE",
            product["Name"],
            flags=re.IGNORECASE,
        )
    )


def get_product_date(product):
    match = re.search(
        r"_(\d{12})_GLOBE",
        product["Name"],
    )

    if not match:
        raise ValueError(
            f"Could not extract date from {product['Name']}"
        )

    return match.group(1)


def get_product_version(product):
    match = re.search(
        r"_V(\d+)\.(\d+)\.(\d+)",
        product["Name"],
    )

    if match:
        return tuple(map(int, match.groups()))

    return (0, 0, 0)


def get_rt_rank(product):
    """
    Prefer the final non-RT archive.

    If no final archive exists for that date, prefer:
    RT6 > RT2 > RT1 > RT0.
    """
    match = re.search(
        r"LAI-RT(\d+)_",
        product["Name"],
        flags=re.IGNORECASE,
    )

    if match:
        return int(match.group(1))

    return 100


products = [
    product
    for product in products
    if is_lai_product(product)
]

print(
    f"LAI products after removing ancillary layers: "
    f"{len(products)}"
)

best_products = {}

for product in products:
    date = get_product_date(product)

    product_score = (
        get_rt_rank(product),
        get_product_version(product),
    )

    if date not in best_products:
        best_products[date] = product
        continue

    current_product = best_products[date]

    current_score = (
        get_rt_rank(current_product),
        get_product_version(current_product),
    )

    if product_score > current_score:
        best_products[date] = product

products = sorted(
    best_products.values(),
    key=get_product_date,
)

print(
    f"Selected {len(products)} unique "
    f"10-daily dates"
)


def stream_name(product):
    match = re.search(
        r"LAI-RT(\d+)_",
        product["Name"],
        flags=re.IGNORECASE,
    )

    if match:
        return f"RT{match.group(1)}"

    return "final archive"


stream_counts = Counter(
    stream_name(product)
    for product in products
)

print("\nSelected product streams:")

for stream, count in sorted(stream_counts.items()):
    print(f"  {stream}: {count}")


# ============================================================
# CONNECT TO COPERNICUS S3
# ============================================================

# Credentials are read from environment variables if available.
# Otherwise, they are entered securely without being written
# into this script.
access_key = os.environ.get("CDSE_S3_ACCESS_KEY")
secret_key = os.environ.get("CDSE_S3_SECRET_KEY")

if not access_key:
    access_key = getpass("CDSE S3 access key: ")

if not secret_key:
    secret_key = getpass("CDSE S3 secret key: ")

s3 = boto3.client(
    "s3",
    endpoint_url="https://eodata.dataspace.copernicus.eu",
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
    region_name="default",
    config=Config(
        signature_version="s3v4",
        s3={"addressing_style": "path"},
        retries={"max_attempts": 10},
    ),
)


# ============================================================
# S3 HELPER FUNCTIONS
# ============================================================

def get_s3_prefix(s3_path):
    if not s3_path:
        raise ValueError("Product has no S3Path")

    prefix = s3_path.lstrip("/")

    if prefix.startswith("eodata/"):
        prefix = prefix[len("eodata/"):]

    return prefix.rstrip("/") + "/"


def find_lai_file(prefix):
    """
    Find the actual LAI TIFF inside the selected product folder.
    """
    candidates = []
    paginator = s3.get_paginator("list_objects_v2")

    for page in paginator.paginate(
        Bucket="eodata",
        Prefix=prefix,
    ):
        for item in page.get("Contents", []):
            filename = Path(item["Key"]).name
            filename_upper = filename.upper()

            if not filename.lower().endswith(
                (".tif", ".tiff")
            ):
                continue

            if any(
                layer in filename_upper
                for layer in [
                    "-RMSE_",
                    "-QFLAG_",
                    "-NOBS_",
                    "-LENGTH_BEFORE_",
                    "-LENGTH_AFTER_",
                    "-LBEFORE_",
                    "-LAFTER_",
                ]
            ):
                continue

            candidates.append(
                {
                    "key": item["Key"],
                    "size": item["Size"],
                    "filename": filename,
                }
            )

    if not candidates:
        raise FileNotFoundError(
            f"No LAI TIFF found under {prefix}"
        )

    candidates.sort(
        key=lambda item: (
            "LAI-LAI_"
            not in item["filename"].upper(),
            "LAI-RT"
            not in item["filename"].upper(),
            item["filename"],
        )
    )

    selected = candidates[0]

    return selected["key"], selected["size"]


# ============================================================
# DOWNLOAD THE SELECTED LAI TIFFS
# ============================================================

local_records = []

for number, product in enumerate(
    products,
    start=1,
):
    timestamp = get_product_date(product)

    year = int(timestamp[:4])
    month = int(timestamp[4:6])
    day = int(timestamp[6:8])

    prefix = get_s3_prefix(product["S3Path"])
    s3_key, remote_size = find_lai_file(prefix)

    product_directory = (
        DOWNLOAD_DIR / product["Name"]
    )

    product_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    local_path = (
        product_directory / Path(s3_key).name
    )

    complete_local_file = (
        local_path.exists()
        and local_path.stat().st_size == remote_size
    )

    if complete_local_file:
        print(
            f"[{number}/{len(products)}] "
            f"Already downloaded: {local_path.name}"
        )
    else:
        if local_path.exists():
            print(
                f"[{number}/{len(products)}] "
                f"Incomplete file found; downloading again: "
                f"{local_path.name}"
            )
        else:
            print(
                f"[{number}/{len(products)}] "
                f"Downloading: {local_path.name}"
            )

        s3.download_file(
            Bucket="eodata",
            Key=s3_key,
            Filename=str(local_path),
        )

    local_records.append(
        {
            "year": year,
            "month": month,
            "day": day,
            "stream": stream_name(product),
            "path": local_path,
        }
    )


if not local_records:
    raise RuntimeError(
        "No LAI files were selected or downloaded"
    )


# ============================================================
# GROUP INPUT FILES BY CALENDAR MONTH
# ============================================================

files_by_month = defaultdict(list)

for record in local_records:
    files_by_month[record["month"]].append(
        record["path"]
    )

print("\nInput maps per calendar month:")

for month in range(1, 13):
    print(
        f"  Month {month:02d}: "
        f"{len(files_by_month[month])} maps"
    )


# ============================================================
# READ AND VALIDATE THE SOURCE GRID
# ============================================================

template_path = local_records[0]["path"]

with rasterio.open(template_path) as template:
    source_width = template.width
    source_height = template.height
    source_transform = template.transform
    source_crs = template.crs
    source_bounds = template.bounds
    source_dtype = template.dtypes[0]
    source_scale = template.scales[0]

if source_crs is None:
    raise ValueError(
        "The source LAI raster has no CRS"
    )

if source_crs.to_epsg() != 4326:
    raise ValueError(
        f"Expected EPSG:4326, found {source_crs}"
    )

if source_dtype != "uint8":
    raise TypeError(
        f"Expected uint8 LAI data, found {source_dtype}"
    )

# Use the documented LAI scale if it is not stored in the COG.
if (
    source_scale is None
    or np.isclose(source_scale, 1.0)
):
    source_scale = 1.0 / 30.0

print("\nSource raster grid:")
print(f"  Width: {source_width}")
print(f"  Height: {source_height}")
print(f"  Bounds: {source_bounds}")
print(f"  CRS: {source_crs}")
print(f"  Data type: {source_dtype}")
print(f"  LAI scale factor: {source_scale}")


for number, record in enumerate(
    local_records,
    start=1,
):
    with rasterio.open(record["path"]) as source:
        same_grid = (
            source.width == source_width
            and source.height == source_height
            and source.transform == source_transform
            and source.crs == source_crs
        )

        if not same_grid:
            raise ValueError(
                "Raster grid mismatch:\n"
                f"{record['path']}"
            )

    if (
        number == 1
        or number % 20 == 0
        or number == len(local_records)
    ):
        print(
            f"Checked source grid "
            f"{number}/{len(local_records)}"
        )


# ============================================================
# DEFINE THE 0.1-DEGREE OUTPUT GRID
# ============================================================

resolution = OUTPUT_RESOLUTION

output_left = (
    np.floor(source_bounds.left / resolution)
    * resolution
)

output_right = (
    np.ceil(source_bounds.right / resolution)
    * resolution
)

output_bottom = (
    np.floor(source_bounds.bottom / resolution)
    * resolution
)

output_top = (
    np.ceil(source_bounds.top / resolution)
    * resolution
)

output_width = int(
    round(
        (output_right - output_left)
        / resolution
    )
)

output_height = int(
    round(
        (output_top - output_bottom)
        / resolution
    )
)

output_transform = from_origin(
    output_left,
    output_top,
    resolution,
    resolution,
)

longitude = (
    output_left
    + (np.arange(output_width) + 0.5)
    * resolution
).astype(np.float64)

latitude = (
    output_top
    - (np.arange(output_height) + 0.5)
    * resolution
).astype(np.float64)

print("\nOutput grid:")
print(f"  Resolution: {resolution}°")
print(f"  Width: {output_width}")
print(f"  Height: {output_height}")
print(
    f"  Bounds: "
    f"{output_left}, {output_bottom}, "
    f"{output_right}, {output_top}"
)


# ============================================================
# CREATE THE 0.1-DEGREE MONTHLY CLIMATOLOGY
# ============================================================

print(f"\nCreating {OUTPUT_PATH}")

with Dataset(
    OUTPUT_PATH,
    mode="w",
    format="NETCDF4",
) as output:

    output.createDimension("month", 12)
    output.createDimension(
        "latitude",
        output_height,
    )
    output.createDimension(
        "longitude",
        output_width,
    )

    month_variable = output.createVariable(
        "month",
        "i1",
        ("month",),
    )

    latitude_variable = output.createVariable(
        "latitude",
        "f8",
        ("latitude",),
    )

    longitude_variable = output.createVariable(
        "longitude",
        "f8",
        ("longitude",),
    )

    vai_variable = output.createVariable(
        "VAI",
        "f4",
        (
            "month",
            "latitude",
            "longitude",
        ),
        fill_value=np.float32(-9999),
        zlib=True,
        complevel=4,
        shuffle=True,
        chunksizes=(
            1,
            min(180, output_height),
            min(360, output_width),
        ),
    )

    month_variable[:] = np.arange(1, 13)
    latitude_variable[:] = latitude
    longitude_variable[:] = longitude

    month_variable.long_name = "calendar month"
    month_variable.month_names = (
        "January February March April May June "
        "July August September October "
        "November December"
    )

    latitude_variable.standard_name = "latitude"
    latitude_variable.long_name = "latitude"
    latitude_variable.units = "degrees_north"
    latitude_variable.axis = "Y"

    longitude_variable.standard_name = "longitude"
    longitude_variable.long_name = "longitude"
    longitude_variable.units = "degrees_east"
    longitude_variable.axis = "X"

    vai_variable.long_name = (
        "Monthly mean leaf area index climatology"
    )
    vai_variable.units = "m2 m-2"
    vai_variable.valid_min = np.float32(0)
    vai_variable.valid_max = np.float32(7)
    vai_variable.spatial_resolution = "0.1 degrees"

    output.title = (
        "Global monthly LAI climatology "
        "for 2018-2020 at 0.1-degree resolution"
    )

    output.source = (
        "Copernicus Global Land Service LAI, "
        "1 km, 10-daily, version 2"
    )

    output.temporal_coverage_requested = (
        "2018-01-01 to 2020-12-31"
    )

    output.spatial_aggregation = (
        "Average of valid native-resolution LAI pixels "
        "overlapping each 0.1-degree cell"
    )

    output.crs = "EPSG:4326"

    maps_per_month = [
        len(files_by_month[month])
        for month in range(1, 13)
    ]

    vai_variable.input_maps_per_month = (
        ",".join(map(str, maps_per_month))
    )

    for month in range(1, 13):
        month_files = files_by_month[month]

        if not month_files:
            print(
                f"No input files for month {month:02d}"
            )
            continue

        print(
            f"\nAggregating month {month:02d} "
            f"from {len(month_files)} maps"
        )

        monthly_sum = np.zeros(
            (output_height, output_width),
            dtype=np.float32,
        )

        monthly_count = np.zeros(
            (output_height, output_width),
            dtype=np.uint8,
        )

        for file_number, path in enumerate(
            month_files,
            start=1,
        ):
            print(
                f"  Map {file_number}/"
                f"{len(month_files)}: "
                f"{path.name}"
            )

            with rasterio.open(path) as source:
                # Read the native uint8 digital numbers.
                raw = source.read(1)

                # Valid LAI digital numbers are 0–210.
                # Larger values are missing-data flags.
                raw[raw > 210] = 255

                aggregated = np.full(
                    (
                        output_height,
                        output_width,
                    ),
                    np.nan,
                    dtype=np.float32,
                )

                reproject(
                    source=raw,
                    destination=aggregated,
                    src_transform=source.transform,
                    src_crs=source.crs,
                    src_nodata=255,
                    dst_transform=output_transform,
                    dst_crs="EPSG:4326",
                    dst_nodata=np.nan,
                    resampling=Resampling.average,
                    init_dest_nodata=True,
                    num_threads=2,
                    warp_mem_limit=512,
                )

            # Convert averaged digital numbers to physical LAI.
            aggregated *= source_scale

            valid = (
                np.isfinite(aggregated)
                & (aggregated >= 0)
                & (aggregated <= 7)
            )

            monthly_sum[valid] += aggregated[valid]
            monthly_count[valid] += 1

            print(
                f"    Native array memory: "
                f"{raw.nbytes / 1024**3:.2f} GB"
            )

            del raw
            del aggregated
            gc.collect()

        monthly_mean = np.full(
            (output_height, output_width),
            -9999,
            dtype=np.float32,
        )

        valid_month = monthly_count > 0

        monthly_mean[valid_month] = (
            monthly_sum[valid_month]
            / monthly_count[valid_month]
        )

        vai_variable[
            month - 1,
            :,
            :,
        ] = monthly_mean

        valid_values = monthly_mean[valid_month]

        if valid_values.size:
            print(
                f"  Monthly mean range: "
                f"{valid_values.min():.3f} to "
                f"{valid_values.max():.3f}"
            )

        del monthly_sum
        del monthly_count
        del monthly_mean
        gc.collect()


print(f"\nFinished: {OUTPUT_PATH}")

Catalogue returned 351 products
LAI products after removing ancillary layers: 351
Selected 90 unique 10-daily dates

Selected product streams:
  RT0: 1
  RT1: 1
  RT2: 4
  RT6: 84


CDSE S3 access key:  ········
CDSE S3 secret key:  ········


[1/90] Already downloaded: c_gls_LAI-LAI-RT6_201801100000_GLOBE_PROBAV_V2.0.1.tiff
[2/90] Already downloaded: c_gls_LAI-LAI-RT6_201801200000_GLOBE_PROBAV_V2.0.1.tiff
[3/90] Already downloaded: c_gls_LAI-LAI-RT6_201801310000_GLOBE_PROBAV_V2.0.1.tiff
[4/90] Already downloaded: c_gls_LAI-LAI-RT6_201802100000_GLOBE_PROBAV_V2.0.1.tiff
[5/90] Already downloaded: c_gls_LAI-LAI-RT6_201802200000_GLOBE_PROBAV_V2.0.1.tiff
[6/90] Already downloaded: c_gls_LAI-LAI-RT6_201802280000_GLOBE_PROBAV_V2.0.1.tiff
[7/90] Already downloaded: c_gls_LAI-LAI-RT6_201803100000_GLOBE_PROBAV_V2.0.1.tiff
[8/90] Already downloaded: c_gls_LAI-LAI-RT6_201803200000_GLOBE_PROBAV_V2.0.1.tiff
[9/90] Already downloaded: c_gls_LAI-LAI-RT6_201803310000_GLOBE_PROBAV_V2.0.1.tiff
[10/90] Already downloaded: c_gls_LAI-LAI-RT6_201804100000_GLOBE_PROBAV_V2.0.1.tiff
[11/90] Already downloaded: c_gls_LAI-LAI-RT6_201804200000_GLOBE_PROBAV_V2.0.1.tiff
[12/90] Already downloaded: c_gls_LAI-LAI-RT6_201804300000_GLOBE_PROBAV_V2.0.1.tiff
[